In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import geopandas as gpd
from shapely import wkt
from shapely.geometry import box
from shapely.geometry import Point
from shapely.ops import nearest_points
from scipy.spatial import cKDTree
from sklearn.impute import SimpleImputer

from osgeo import gdal
from scipy.interpolate import griddata
from scipy.spatial import distance
from scipy.spatial import cKDTree
from pyproj import CRS, Proj, Transformer
from shapely.geometry import box

from datetime import datetime, timedelta

import xarray as xr
import os
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

In [2]:
def mean_with_nan(arrays):
    """
    Calculate the mean of multiple 2D arrays considering NaN values.
    If one element is NaN and others have values, the result takes the non-NaN value(s).
    """
    stacked_arrays = np.stack(arrays)
    
    sum_array = np.nansum(stacked_arrays, axis=0)
    
    count_non_nan = np.sum(~np.isnan(stacked_arrays), axis=0).astype(float)
    
    count_non_nan[count_non_nan == 0] = np.nan
    
    mean_array = sum_array / count_non_nan
    
    return mean_array

# Base grids - 1km

In [3]:
grid = pd.read_csv('grid_proj_1km.csv')[['x', 'y', 'geometry', 'lon', 'lat']]

grid['geometry'] = grid['geometry'].apply(wkt.loads)

grid_gdf = gpd.GeoDataFrame(grid, geometry='geometry', crs='ESRI:102003')

In [4]:
states = gpd.read_file('us_boundaries/States/conus_states.shp')
five_states = states.loc[states['STUSPS'].isin(['NY','PA','OH','MD','WV'])]

polygon = box(-84, 37, -74, 43.5)
region = five_states.clip(polygon)
region_proj = region.to_crs('ESRI:102003')

res = 1000

# Get x and y extent
min_x, min_y, max_x, max_y = region_proj.total_bounds

# Create gridcells with the target resolution
xnew = np.arange(min_x, max_x, res)
ynew = np.arange(min_y, max_y, res)

# Compose two 2D arrays
# xx: x array
# yy: y array
x_1km, y_1km = np.meshgrid(xnew,ynew)

# Create a transformer to convert from projected CRS to geographic CRS
transformer = Transformer.from_crs("ESRI:102003", "EPSG:4326", always_xy=True)

# Transform the coordinates from x, y (projected) to lon, lat (geographic)
lon, lat = transformer.transform(x_1km, y_1km)

# Define the bounding box from the target grid
min_lon = lon.min()
max_lon = lon.max()
min_lat = lat.min()
max_lat = lat.max()

# Batch process TROPOMI so2 to 1km grids

In [ ]:
tropomi_so2_path = 'TROPOMI SO2/'

files = [f for f in os.listdir('TROPOMI_vs_SO2/') if f.endswith('csv')]
files.sort()

In [ ]:
for f in files:

    date = f.split('.')[0]
    print(date)
    
    if not os.path.exists('TROPOMI_vs_SO2/1km_grid/'+date+'.csv'):
    
        year, month, day = date[:4], date[5:7], date[8:]
        date_ = year + month + day
        tropomi_so2_files = [f for f in os.listdir(tropomi_so2_path) if date_ in f]
        
        so2_grids_list = []
        
        if len(tropomi_so2_files) > 1: 
    
            for tropomi_so2_file in tropomi_so2_files:

                try:

                    ds = xr.open_dataset(tropomi_so2_path + tropomi_so2_file,
                                    group='PRODUCT')

                    ds = ds.rio.write_crs(4326)
                    
                    obs_time = ds.time.values.astype(str)[0]

                    lon = ds['longitude'].rio.interpolate_na()[0,:,:]
                    lat = ds['latitude'].rio.interpolate_na()[0,:,:]

                    so2_converted = ds['sulfurdioxide_total_vertical_column'] * ds['sulfurdioxide_total_vertical_column'].multiplication_factor_to_convert_to_molecules_percm2 

                    so2_converted = so2_converted.where(ds['qa_value']>0.5, np.nan)

                    so2_converted = so2_converted.where(so2_converted >= 0, np.nan)

                    so2_converted = so2_converted[0,:,:]

                    flat_lats = ds['latitude'].values.flatten()
                    flat_lons = ds['longitude'].values.flatten()
                    flat_so2 = so2_converted.values.flatten()

                    mask = ~np.isnan(flat_so2)

                    masked_lats = flat_lats[mask]
                    masked_lons = flat_lons[mask]
                    masked_so2 = flat_so2[mask]

                    lat_da = xr.DataArray(masked_lats, dims='points')
                    lon_da = xr.DataArray(masked_lons, dims='points')
                    so2_da = xr.DataArray(masked_so2, dims='points', coords={'lat': lat_da, 'lon': lon_da})
                    
                    tree = cKDTree(np.c_[masked_lats, masked_lons])
                    
                    query_points = np.c_[grid_gdf['lat'].values, grid_gdf['lon'].values]
                    
                    distances, indices = tree.query(query_points, k=1)

                    selected_so2_values = np.full_like(grid_gdf['lat'].values, np.nan, dtype=np.float64)

                    valid_mask = distances < 0.07

                    valid_indices = indices[valid_mask]

                    selected_so2_values[valid_mask] = masked_so2[valid_indices]

                    so2_grids_list.append(selected_so2_values)

                except:
                    print(tropomi_so2_file, 'not readable')
                    
        if len(so2_grids_list) > 0:
        
            grid_gdf['so2'] = mean_with_nan(so2_grids_list)
            
            grid_gdf[['so2']].to_csv('TROPOMI_vs_SO2/1km_grid/'+date+'.csv', index=False)
    
        else:
            print('no data')

# Base grids - 5km

In [3]:
grid = pd.read_csv('grid_proj_5km.csv')[['x', 'y', 'geometry', 'lon', 'lat']]

grid['geometry'] = grid['geometry'].apply(wkt.loads)

grid_gdf = gpd.GeoDataFrame(grid, geometry='geometry', crs='ESRI:102003')

In [4]:
states = gpd.read_file('us_boundaries/States/conus_states.shp')
five_states = states.loc[states['STUSPS'].isin(['NY','PA','OH','MD','WV'])]

polygon = box(-84, 37, -74, 43.5)
region = five_states.clip(polygon)
region_proj = region.to_crs('ESRI:102003')

res = 5000

min_x, min_y, max_x, max_y = region_proj.total_bounds

xnew = np.arange(min_x, max_x, res)
ynew = np.arange(min_y, max_y, res)

x_5km, y_5km = np.meshgrid(xnew,ynew)

transformer = Transformer.from_crs("ESRI:102003", "EPSG:4326", always_xy=True)

lon, lat = transformer.transform(x_5km, y_5km)

min_lon = lon.min()
max_lon = lon.max()
min_lat = lat.min()
max_lat = lat.max()

# Batch process TROPOMI so2 to 5km grids

In [ ]:
tropomi_so2_path = 'TROPOMI SO2/'

files = [f for f in os.listdir('TROPOMI_vs_SO2/') if f.endswith('csv')]
files.sort()

In [ ]:
for f in files:

    date = f.split('.')[0]
    print(date)
    
    if not os.path.exists('TROPOMI_vs_SO2/5km_grid/'+date+'.csv'):
    
        year, month, day = date[:4], date[5:7], date[8:]
        date_ = year + month + day
        tropomi_so2_files = [f for f in os.listdir(tropomi_so2_path) if date_ in f]
        
        so2_grids_list = []
        
        if len(tropomi_so2_files) > 1: 
    
            for tropomi_so2_file in tropomi_so2_files:

                try:

                    ds = xr.open_dataset(tropomi_so2_path + tropomi_so2_file,
                                    group='PRODUCT')

                    ds = ds.rio.write_crs(4326)
                    
                    obs_time = ds.time.values.astype(str)[0]

                    lon = ds['longitude'].rio.interpolate_na()[0,:,:]
                    lat = ds['latitude'].rio.interpolate_na()[0,:,:]

                    so2_converted = ds['sulfurdioxide_total_vertical_column'] * ds['sulfurdioxide_total_vertical_column'].multiplication_factor_to_convert_to_molecules_percm2 

                    so2_converted = so2_converted.where(ds['qa_value']>0.5, np.nan)

                    so2_converted = so2_converted.where(so2_converted >= 0, np.nan)

                    so2_converted = so2_converted[0,:,:]

                    flat_lats = ds['latitude'].values.flatten()
                    flat_lons = ds['longitude'].values.flatten()
                    flat_so2 = so2_converted.values.flatten()

                    mask = ~np.isnan(flat_so2)

                    masked_lats = flat_lats[mask]
                    masked_lons = flat_lons[mask]
                    masked_so2 = flat_so2[mask]

                    lat_da = xr.DataArray(masked_lats, dims='points')
                    lon_da = xr.DataArray(masked_lons, dims='points')
                    so2_da = xr.DataArray(masked_so2, dims='points', coords={'lat': lat_da, 'lon': lon_da})
                    
                    tree = cKDTree(np.c_[masked_lats, masked_lons])
                    
                    query_points = np.c_[grid_gdf['lat'].values, grid_gdf['lon'].values]
                    
                    distances, indices = tree.query(query_points, k=1)

                    selected_so2_values = np.full_like(grid_gdf['lat'].values, np.nan, dtype=np.float64)

                    valid_mask = distances < 0.07

                    valid_indices = indices[valid_mask]

                    selected_so2_values[valid_mask] = masked_so2[valid_indices]

                    so2_grids_list.append(selected_so2_values)

                except:
                    print(tropomi_so2_file, 'not readable')
                    
        if len(so2_grids_list) > 0:
        
            grid_gdf['so2'] = mean_with_nan(so2_grids_list)
            
            grid_gdf[['so2']].to_csv('TROPOMI_vs_SO2/5km_grid/'+date+'.csv', index=False)
    
        else:
            print('no data')